In [7]:
from pathlib import Path
from sqlalchemy import create_engine

# Build the path and engine
db_path = Path.cwd().parent / 'data' / 'database' / 'retail.db'
engine = create_engine(f"sqlite:///{db_path}")

# Load the extension and connect using the engine object directly
%load_ext sql
%sql engine

# Confirm connection and check what tables exist
%sql SELECT name FROM sqlite_master WHERE type='table'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

name
retail_turnover


### Query 1: YoY Growth by Category

💡 Which retail categorie grew faster or slower than last year, and by how much?

In [20]:
%%sql
-- Create helping column annual which aggregates sales per category per year
WITH annual AS (
    SELECT
        year,
        category,
        SUM(turnover_m) AS turnover_m
    FROM retail_turnover
    WHERE category != 'Total (Industry)' -- exclude the aggregate row
    GROUP BY year, category
)
SELECT
    a.year,
    a.category,
    ROUND(a.turnover_m, 1) AS turnover_m,
    ROUND(b.turnover_m, 1) AS prior_year_m,
    ROUND((a.turnover_m - b.turnover_m) / b.turnover_m * 100, 1) AS yoy_growth_pct
FROM annual a
JOIN annual b 
    ON a.category = b.category
    AND a.year = b.year + 1
ORDER BY a.year DESC, yoy_growth_pct DESC; -- best performers first

Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

year,category,turnover_m,prior_year_m,yoy_growth_pct
2025,Food retailing,86803.5,173604.2,-50.0
2025,"Cafes, restaurants and takeaway food services",32272.3,65411.7,-50.7
2025,Other retailing,33384.1,68326.2,-51.1
2025,Household goods retailing,33634.6,70416.0,-52.2
2025,"Clothing, footwear and personal accessory retailing",17218.9,36146.3,-52.4
2025,Department stores,10518.9,22853.9,-54.0
2024,Other retailing,68326.2,64758.1,5.5
2024,Food retailing,173604.2,168448.9,3.1
2024,"Cafes, restaurants and takeaway food services",65411.7,63985.5,2.2
2024,"Clothing, footwear and personal accessory retailing",36146.3,35610.7,1.5


### Query 2: Current Underperformers vs National Benchmark

💡 Which categories are growing slower than the national total right now, over the past 12 months?


In [12]:
%%sql
-- Helper column recent which summates the most recent 12 months of turnover per category
WITH recent AS (
    SELECT
        category, 
        SUM(turnover_m) AS turnover_12m
    FROM retail_turnover
    WHERE date >= date('now', '-12 months')
    GROUP BY category
),

-- Helper column prior which summates the 12 months before the most recent period (the comparison period)
prior AS (
    SELECT 
        category, 
        SUM(turnover_m) AS turnover_prior_12m
    FROM retail_turnover
    WHERE date >= date('now', '-24 months')
    AND date < date('now', '-12 months')
    GROUP BY category
)

SELECT
    r.category,
    ROUND(r.turnover_12m, 1) AS last_12m_m,
    ROUND((r.turnover_12m - p.turnover_prior_12m) / p.turnover_prior_12m * 100, 1) AS yoy_growth_pct
FROM recent r
JOIN prior p 
    ON r.category = p.category
WHERE r.category != 'Total (Industry)'
ORDER BY yoy_growth_pct ASC; -- worst performers at the top


Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

category,last_12m_m,yoy_growth_pct
Household goods retailing,17185.5,-75.7
Department stores,5625.2,-75.4
Other retailing,17023.9,-75.4
"Cafes, restaurants and takeaway food services",16385.6,-75.1
"Clothing, footwear and personal accessory retailing",9048.5,-75.1
Food retailing,43372.8,-75.1
